![alt text](download.png)

pages:

`! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain`

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("apikey.env")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = "RAG QUERY TANSLATION"

In [2]:
from langchain_deepseek import ChatDeepSeek
from langchain_huggingface import HuggingFaceEmbeddings

BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')

deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")

model = ChatDeepSeek(temperature=0.0,
                     api_key=API_KEY, 
                     base_url=BASE_URL, 
                     model=deepseek_chat_model)
embeddings = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")

SECESSFULLY!


In [3]:
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS

raw_documents = DirectoryLoader('./state2/example/corpus/', 
                                glob="**/*.txt",
                                show_progress=True,
                                use_multithreading=True,
                                loader_cls=TextLoader,
                                loader_kwargs={
                                    "encoding": "utf-8",
                                }
                                )
load_doc = raw_documents.lazy_load()

text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=50)
documents = text_splitter.split_documents(list(load_doc))

store = LocalFileStore("./cache/FourGreatClasissDB")
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings, 
    store, 
    namespace=embeddings.model_name
)
simpleDB = FAISS.from_documents(documents, cached_embedder)

# 检索器
retriever = simpleDB.as_retriever() 

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 96.50it/s]
Created a chunk of size 2625, which is longer than the specified 2000
Created a chunk of size 2595, which is longer than the specified 2000
Created a chunk of size 2905, which is longer than the specified 2000
Created a chunk of size 2176, which is longer than the specified 2000
Created a chunk of size 2025, which is longer than the specified 2000
Created a chunk of size 2609, which is longer than the specified 2000
Created a chunk of size 2163, which is longer than the specified 2000
Created a chunk of size 2343, which is longer than the specified 2000
Created a chunk of size 2502, which is longer than the specified 2000
Created a chunk of size 2190, which is longer than the specified 2000
Created a chunk of size 2590, which is longer than the specified 2000
Created a chunk of size 2622, which is longer than the specified 2000
Created a chunk of size 3339, which is l

## 1. RAG-FUSION

In [41]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


template = """
你是一个得力的助手，擅长从用户的单个输出(query)生成多种丰富的搜索问题的内容。
并且生成的内容和{qustion}高度相关，\n
输出四个查查询，格式为List[str]
"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generation_queries = (
    prompt_rag_fusion # | RunnablePassthrough()
    | model
    | JsonOutputParser()
)

In [57]:
question = "在西游记中，唐僧和谁去西天取经？"

out = generation_queries.invoke(question)
out, type(out)

(['唐僧西天取经的徒弟有哪些人', '西游记中陪同唐僧取经的主要角色', '唐僧取经团队的主要成员介绍', '孙悟空猪八戒沙僧与唐僧的关系'], list)

In [43]:
from langchain.load import dumps, loads

# 多检索融合
def reciprocal_rank_fusion(results: list[list], k=60, max_docs=50):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results[:max_docs]

In [ ]:
retrival_chain_fusion = (generation_queries
                        | retriever.map()  #可连接在上游链条中并行 map 每个输入元素
                        | reciprocal_rank_fusion)

docs = retrival_chain_fusion.invoke({question})
len(docs)

In [56]:
template = """根据给出的上下文回答下面的问题：
{context}
问题: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

full_chain = (RunnableLambda(lambda x: {
                "context": retrival_chain_fusion.invoke(x),  # 串行执行
                "question": x
            })
            | prompt 
            | model)

for chunk in full_chain.stream({question}):
    print(chunk.content, end="", flush=True)

根据《西游记》原文内容，唐僧前往西天取经时，与三位徒弟同行，他们分别是：

1. **孙悟空**（孙行者）  
2. **猪悟能**（猪八戒）  
3. **沙悟净**（沙和尚）  

此外，还有一匹由西海龙王之子化身的**白马**作为脚力。唐僧师徒四人历经九九八十一难，最终抵达西天取得真经。

## part2 Decomposition


In [84]:
from langchain.prompts import ChatPromptTemplate

# Decomposition
template = """你是一个帮助性助手，负责生成与输入问题相关的多个子问题。  
目标是将输入问题分解为一系列可以独立回答的子问题。  
生成与以下内容相关的多个搜索查询：: {question}\n 
输出为三个查询:"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [85]:
generate_queries_decomposition = ( prompt_decomposition 
                                  | model 
                                  | StrOutputParser() 
                                  | (lambda x: x.split("\n")))

# Run
question = "菩提祖师教了孙悟空什么本领？"
questions = generate_queries_decomposition.invoke({"question":question})

In [86]:
questions

['1. 菩提祖师传授给孙悟空的七十二变具体内容是什么？  ',
 '2. 孙悟空从菩提祖师处学到的筋斗云有何特点与能力？  ',
 '3. 菩提祖师除七十二变和筋斗云外，还教了孙悟空哪些法术或本领？']

### Answer recursively  

![alt text](download-1.png)

Papers:

* https://arxiv.org/pdf/2205.10625.pdf
* https://arxiv.org/abs/2212.10509.pdf

In [87]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)


def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()


In [88]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | model
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

In [89]:
answer

'根据《西游记》原著内容，菩提祖师除七十二变和筋斗云外，还传授了孙悟空以下法术或本领：\n\n1. **长生不老之术**（根本法门）：  \n   菩提祖师最初传授孙悟空的是长生妙诀，作为修道的根基，使他脱离凡胎，获得寿元。\n\n2. **法术口诀与捻诀运用**：  \n   孙悟空在后续情节中多次使用"捻诀"施法（如呼风唤雨、避火诀等），这些基础法术操控能力源自菩提祖师的系统教导。\n\n3. **腾云驾雾基础**：  \n   在教授筋斗云前，祖师先传授了普通的腾云之法（如"爬云"），后因孙悟空资质特殊才改授筋斗云。\n\n4. **兵器运用与神通根基**：  \n   虽未直接记载兵器训练，但孙悟空后来熟练运用金箍棒、分身法等神通，其根基均与祖师传授的"道法根本"相关。\n\n**依据分析**：  \n- 以上结论基于《西游记》第二回「悟彻菩提真妙理 断魔归本合元神」中祖师传道的整体描写，孙悟空在离开灵台方寸山时已具备综合神通能力。  \n- 具体细节需结合原著前后章节印证，例如孙悟空在取经途中使用的多种法术（如法天象地、身外身法等）均以七十二变和长生术为基础演化而来。  \n\n> 注：菩提祖师所传实为**道家正统修炼体系**，七十二变与筋斗云仅是其中最具标志性的神通，其他基础法门贯穿于孙悟空整体能力体系中。'

### Answer individually 
![alt text](download-2.png)


In [98]:
from langchain import hub
prompt_rag = hub.pull("rlm/rag-prompt")

In [101]:
prompt_rag.pretty_print()

================================ Human Message =================================

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:


In [109]:
# Answer each sub-question individually 

from langchain import hub
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# RAG prompt
prompt_rag = hub.pull("rlm/rag-prompt")
question = "请你较为全面地宋江生平事迹。"

def retrieve_and_rag(question,prompt_rag,sub_question_generator_chain):
    """RAG on each sub-question"""
    
    # Use our decomposition / 
    sub_questions = sub_question_generator_chain.invoke({"question":question})
    
    # Initialize a list to hold RAG chain results
    rag_results = []
    
    for sub_question in sub_questions:
        
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.get_relevant_documents(sub_question)
        
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | model | StrOutputParser()).invoke({"context": retrieved_docs, 
                                                                "question": sub_question})
        rag_results.append(answer)
    
    return rag_results,sub_questions

# Wrap the retrieval and RAG process in a RunnableLambda for integration into a chain
answers, questions = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)

In [110]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | model
    | StrOutputParser()
)

for chunk in final_rag_chain.stream({"context":context,"question":question}):
    print(chunk, end="", flush=True)

基于所提供的问答资料，宋江的生平事迹可综合梳理如下：

### 一、早期经历与上梁山背景
根据现有资料，宋江早期具体生平细节未明确记载，但可知他在上梁山前已是江湖中颇具声望的“及时雨”。其被迫落草的经历与官府压迫、仗义助人等事件相关，最终成为梁山起义军的核心人物。

### 二、梁山时期的领导与军事成就
1. **聚义领导**：宋江继晁盖后成为梁山首领，重整山寨秩序，确立“替天行道”纲领，通过排定座次整合各方力量。  
2. **重要战役**：  
   - **兵打大名府**：为营救卢俊义，率军攻破大名府，展现战略魄力。  
   - **抗辽与平乱**：招安前率梁山军队抗击外敌、平定地方叛乱，巩固了梁山势力。  
3. **招安决策**：主动接受朝廷招安，将梁山队伍转化为官方军队，这一选择体现了其“忠君报国”的思想，但也成为生涯转折点。

### 三、招安后的结局与历史评价
1. **征讨方腊**：招安后奉命南征方腊，虽智取润州等战役获胜，但梁山好汉伤亡惨重，实力大幅削弱。  
2. **悲剧结局**：战后受封“忠烈义济灵应侯”，却遭奸臣妒忌，被毒酒害死，最终魂聚蓼儿洼。  
3. **历史评价**：  
   - **正面**：强调其忠义双全，以招安实现“报效朝廷”的理想；  
   - **批判**：认为招安导致梁山起义精神消亡，队伍结局惨淡，折射出封建时代抗争者的局限性。

### 总结
宋江的一生从江湖豪杰至梁山领袖，再到招安将领，其事迹交织着反抗与妥协、忠义与悲情。他的形象既反映了传统社会对“忠君”道德的推崇，也揭示了农民起义在历史洪流中的复杂命运。

## step-back prompting
![alt text](download-3.png)

In [ ]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

generate_queries_step_back = prompt | model | StrOutputParser()

In [ ]:
question = "红楼梦的《葬花吟》是在哪个经典剧情里面出现的？"
for chunk in generate_queries_step_back.stream({"question": question}):
    print(chunk, end="", flush=True)

《红楼梦》中的经典情节有哪些？

In [116]:
# Response prompt 
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | model
    | StrOutputParser()
)

for chunk in chain.stream({"question": question}):
    print(chunk, end="", flush=True)

《葬花吟》出现在《红楼梦》第二十七回“滴翠亭杨妃戏彩蝶 埋香冢飞燕泣残红”的经典剧情中。该回中，林黛玉因前一天晚上去怡红院探望贾宝玉时，被晴雯误拒门外，心生误会和伤感。次日恰逢饯花之期，黛玉见落花满地，触景生情，于是在大观园内的“埋香冢”附近，一边葬花，一边吟诵了《葬花吟》。贾宝玉在山坡上听到后，尤其是“侬今葬花人笑痴，他年葬侬知是谁”“一朝春尽红颜老，花落人亡两不知”等句，感极而悲，恸倒在山坡上。这一情节深刻展现了黛玉的多愁善感和悲剧命运，是《红楼梦》中标志性的场景之一。

## HyDE
![alt text](download-4.png)

In [117]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# HyDE document generation
template = """你是一个专业的分析师，请你写一个对于这个问题的调研报告。
Question: {question}
Passage:"""

prompt_hyde = ChatPromptTemplate.from_template(template)


generate_docs_for_retrieval = (
    prompt_hyde | model | StrOutputParser() 
)

# Run
question = "为什么请诸葛亮出山需要三顾茅庐？"
for chunk in generate_docs_for_retrieval.stream({"question":question}):
    print(chunk, end="", flush=True)

好的，作为一名专业分析师，我将对“为什么请诸葛亮出山需要三顾茅庐”这一问题进行深入调研和分析，并为您呈现一份结构化的报告。

---

### **关于“三顾茅庐”事件必要性的分析报告**

**报告摘要**
本报告旨在深度剖析“三顾茅庐”这一历史典故背后的核心动因。通过分析所提供的文本材料，并结合《三国志》等史籍的佐证，我们认为，“三顾茅庐”并非偶然，而是一场由诸葛亮精心策划、刘备全力配合的、双方共同成就的“双向奔赴”。其必要性根植于诸葛亮对自身价值的极致定位、对理想主公的严格筛选，以及刘备对顶级人才无可替代的迫切需求。这一事件不仅确立了诸葛亮“帝王师”的崇高起点，也为蜀汉政权的建立奠定了核心基石。

---

#### **一、 核心原因分析**

“三顾茅庐”的必要性可以从三个主要层面进行解构：

**1. 对诸葛亮而言：一次精准的个人品牌营销与价值确认**

*   **确立“非世俗人才”的顶级人设：** 诸葛亮自比管仲、乐毅，其志向并非普通谋士。如果一请即出，则与寻常求职者无异，无法凸显其稀缺性和独特性。“三顾”的过程，本身就是一场公开的、高规格的品牌宣传，向天下宣告：诸葛亮是值得一方诸侯反复恳求的“卧龙”。
*   **测试刘备的诚意与耐心：** 诸葛亮的才能是“定国安邦”之才，他需要的是一个能绝对信任、彻底放权的君主。通过设置“三顾”的考验，他可以观察刘备是否如传说中那般“仁德”、有耐心、有恒心。这直接关系到未来合作中他能获得多大的决策空间和话语权。
*   **抬高自身政治起点：** 轻易得到的东西往往不被珍惜。通过让刘备付出巨大的时间与诚意成本，诸葛亮确保了自己在出山之初就能获得极高的礼遇和地位，而非从基层幕僚做起。这为他日后在蜀汉集团中“一人之下，万人之上”的地位铺平了道路。

**2. 对刘备而言：一次彰显其政治家风范与迫切需求的表演**

*   **展示求贤若渴的明主形象：** 刘备当时正处于事业低谷，寄人篱下，他最核心的竞争力就是“汉室宗亲”的品牌和“仁德”的名声。“三顾茅庐”是他将这一品牌形象具象化的最佳实践。他通过这一行为向天下英才宣告：我刘备尊重人才，不惜屈尊降贵。这为他后续吸引更多人才（如庞统、法正等）起到了极佳的示范效应。
*   **匹配顶级人才的超高成本：** 刘备深知，要获得能够扭转乾坤的顶级战略家，就必须付出与之匹配的诚

In [118]:
# Retrieve
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(id='1dcc91b7-1b02-4ea8-9574-6540ec3ebbd7', metadata={'source': 'state2\\example\\corpus\\三国演义.txt'}, page_content='操既定大事，乃设宴后堂，聚众谋士共议曰：“刘备屯兵徐州，自领州事；近吕布以兵败投之，备使居于小沛：若二人同心引兵来犯，乃心腹之患也。公等有何妙计可图之？”许褚曰：“愿借精兵五万，斩刘备、吕布之头，献于丞相。”荀彧曰：“将军勇则勇矣，不知用谋。今许都新定，未可造次用兵。彧有一计，名曰二虎竞食之计。今刘备虽领徐州，未得诏命。明公可奏请诏命实授备为徐州牧，因密与一书，教杀吕布。事成则备无猛士为辅，亦渐可图；事不成，则吕布必杀备矣：此乃二虎竞食之计也。”操从其言，即时奏请诏命，遣使赍往徐州，封刘备为征东将军宜城亭侯领徐州牧；并附密书一封。却说刘玄德在徐州，闻帝幸许都，正欲上表庆贺。忽报天使至，出郭迎接入郡，拜受恩命毕，设宴管待来使。使曰：“君侯得此恩命，实曹将军于帝前保荐之力也。”玄德称谢。使者乃取出私书递与玄德。玄德看罢，曰：“此事尚容计议。”席散，安歇来使于馆驿。玄德连夜与众商议此事。张飞曰：“吕布本无义之人，杀之何碍！”玄德曰：“他势穷而来投我，我若杀之，亦是不义。”张飞曰：“好人难做！”玄德不从。次日，吕布来贺，玄德教请入见。布曰：“闻公受朝廷恩命，特来相贺。”玄德逊谢。只见张飞扯剑上厅，要杀吕布。玄德慌忙阻住。布大惊曰：“翼德何故只要杀我？”张飞叫曰：“曹操道你是无义之人，教我哥哥杀你！”玄德连声喝退。乃引吕布同入后堂，实告前因；就将曹操所送密书与吕布看。布看毕，泣曰：“此乃曹贼欲令我二人不和耳！”玄德曰：“兄勿忧，刘备誓不为此不义之事。”吕布再三拜谢。备留布饮酒，至晚方回。关、张曰：“兄长何故不杀吕布？”玄德曰：“此曹孟德恐我与吕布同谋伐之，故用此计，使我两人自相吞并，彼却于中取利。奈何为所使乎？”关公点头道是。张飞曰：“我只要杀此贼以绝后患！”玄德曰：“此非大丈夫之所为也。”\n\n    次日，玄德送使命回京，就拜表谢恩，并回书与曹操，只言容缓图之。使命回见曹操，言玄德不杀吕布之事。操问荀彧曰：“此计不成，奈何？”或曰：“又有一计，名曰驱虎吞狼之计。”操曰：“其计如何？”彧曰：“可暗令人往袁术处通问，报说刘备上密

In [121]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | model
    | StrOutputParser()
)

for chunk in final_rag_chain.stream({"context":retrieved_docs,"question":question}):
    print(chunk, end="", flush=True)

根据提供的上下文，特别是第三个文档中关于刘备三顾茅庐的描述，可以总结出需要“三顾茅庐”的原因如下：

1.  **诸葛亮本人的态度**：诸葛亮自称是“南阳野人，疏懒性成”，并且说“亮久乐耕锄，懒于应世，不能奉命”。这表明他起初是安于隐居生活、不愿出山参与纷繁世事的。因此，前两次的拜访未能见到他，以及他最初的推辞，都反映了他并非轻易就能被请动的人。

2.  **刘备的诚意和决心**：正因为诸葛亮态度谨慎且才能卓越，刘备才需要通过“三顾”来展现自己极大的诚意和求贤若渴的决心。文中描述刘备“泪沾袍袖，衣襟尽湿”，并说“先生不出，如苍生何”，这种至诚的态度最终打动了诸葛亮。

3.  **相互的试探与认可**：这个过程也是双方相互了解和建立信任的机会。诸葛亮在第三次会面中为刘备分析了天下大势，提出了“隆中对”的宏伟战略，这证明了他确实有经天纬地之才。而刘备的再三邀请也向诸葛亮证明了自己是一位值得辅佐的明主。

综上所述，三顾茅庐既是由于诸葛亮本人起初无意出山，也需要刘备展现出足够的诚意来打动这位旷世奇才，同时这也是一个双方奠定未来深厚君臣关系的基础过程。